# B 방식 검색: 조문 검색 + 항 리랭크 (Colab GPU)

2026-11-19에 최종 선정한 **B 방식** 검색만 따로 떼어 정리한 노트북이에요. LLM 답변 생성은 넣지 않았어요.

```
질문
 → ② 하이브리드 검색 (조문 단위 조각)
      Dense(BGE-M3) 30 + BM25(Kiwi, backend rewrite로 규정 용어 보강) 30 → 가중 RRF(법령 ×0.6) → 후보 40조각 → 조문으로 묶음
 → ③ 리랭커 (항 단위)
      후보 조문을 항으로 펼쳐 bge-reranker-v2-m3로 항마다 점수 → 조문 점수 = 그 조문에서 가장 높은 항 점수
 → LLM에 넘길 조문 5개
      조문마다 점수 높은 항부터 이어 붙이고(+조문이 언급한 별표 1개) 1,200자에서 자름
```

| 절 | 내용 |
| --- | --- |
| 1~3 | 설치 · 데이터 · 모델 |
| 4 | B 검색 함수 (`search_B`) |
| 5 | **질문 넣어 보기** (`ask("…")`) |
| 6 | 골든셋 dev·test 지표: 정답 항 전달률 · 조문 recall@5·30 · MRR · 속도 |
| 7 | 거절 기준값(`verify_min_score`)을 dev로 정하고 test로 확인 |

- 데이터는 GitHub `backend`·`DB`·`RAG` main을 받아 써요. 조문 조각은 `DB/data/regs`·`data/laws`, 항 조각은 `DB/data/hang`.
- **런타임 → 런타임 유형 변경 → GPU(T4 이상) → 모두 실행.** T4 기준 10~15분 (임베딩 2,700조각 + 평가 241문항).

## 1. 설치 · 설정

In [ ]:
!pip install -q -U "sentence-transformers>=3.4" rank_bm25 kiwipiepy python-dotenv

In [ ]:
import os, sys, json, re, time, subprocess, importlib.util
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, torch

ROOT = Path("/content/hr_chatbot")
ROOT.mkdir(parents=True, exist_ok=True)

# B 최종 설정 (11/19 비교 노트북 hang_long_article_eval 과 같음)
CFG = {"embed_model": "BAAI/bge-m3", "rerank_model": "BAAI/bge-reranker-v2-m3",
       "k_dense": 30, "k_bm25": 30, "k_candidates": 40, "rrf_c": 60,
       "bm25_weight": 1.0, "dense_weight": 1.0, "law_weight": 0.6, "rewrite_bm25": True,
       "internal_bonus": 0.0,          # 내부규정 가점. 11/18은 0.2였지만 거절 기준을 망가뜨려서 11/19부터 0
       "k_context": 5, "attach_appendix": True, "max_chars_per_article": 1200}

for repo in ("backend", "DB", "RAG"):
    d = ROOT / repo
    if d.exists():
        subprocess.run(f"rm -rf {d}", shell=True)
    subprocess.run(f"git clone -q --depth 1 https://github.com/Jaeyong-s-HR-Chatbot/{repo}.git {d}", shell=True, check=True)
    print(f"{repo:8s}", subprocess.run(f"git -C {d} log -1 --format='%h %ad %s' --date=short", shell=True,
                                       capture_output=True, text=True).stdout.strip())
BACKEND, DB, RAG = ROOT / "backend", ROOT / "DB", ROOT / "RAG"

# 팀 채점 스크립트 (골든셋 읽기 · dev/test 분할)
os.environ["BACKEND_DIR"] = str(BACKEND)
spec = importlib.util.spec_from_file_location("eval_golden", RAG / "scripts/eval_golden.py")
eg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eg)
ITEMS = eg.load_items()
SPLIT = eg.get_splits(ITEMS)

# backend 질의 재작성(glossary) = 설계도 ①의 search_query. DB 없이 불러옴
os.environ.update({"HRDB_URL": "", "SEARCH_MODE": "keyword", "LLM_BASE_URL": "", "PSEUDO_KEY": "colab-notebook-only"})
sys.path.insert(0, str(BACKEND))
from app.graph import retriever as be_retriever
print(f"골든셋 {len(ITEMS)}문항 · 재작성 예: {be_retriever.rewrite('결혼하면 경조휴가 며칠이야?')}")

## 2. 데이터: 조문 조각(검색용) · 항 조각(리랭커용)
- **조문 조각** (`regs`·`laws/chunks.jsonl`): ② 검색 대상. 긴 조문은 여러 조각(part)으로 나뉘어 있어요
- **항 조각** (`hang/chunks.jsonl`): ③ 리랭커가 점수를 매기는 단위
- `old_to_new.json`: 조문 조각 → 조문 번호(`article_id`). 같은 조문의 항들을 찾는 열쇠예요

In [ ]:
def jl(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

def _cite(r):
    art, t = (r.get("article") or "").strip(), (r.get("article_title") or "").strip()
    return f"{r['doc_name']} {art}" + (f"({t[:20]})" if t and art.startswith("제") else "")

def load_corpus(paths):
    out = []
    for p in paths:
        for r in jl(p):
            if not r.get("searchable"):
                continue
            out.append({"id": r["chunk_id"], "doc_id": r["document_id"], "doc_name": r["doc_name"],
                        "doc_kind": r.get("doc_kind") or ("법령" if r["chunk_id"].startswith("L") else "내부규정"),
                        "article": (r.get("article") or "").strip(), "article_title": (r.get("article_title") or "").strip(),
                        "part": r.get("part") or 1, "citation": _cite(r), "revised": r.get("effective_date") or "",
                        "text": r["text"], "article_id": r.get("article_id"),
                        "embed_text": f"{r['doc_name']} {r.get('article') or ''} {r.get('article_title') or ''}".strip() + "\n" + r["text"]})
    return out

OLDC = load_corpus([DB / "data/regs/chunks.jsonl", DB / "data/laws/chunks.jsonl"])   # 조문 조각
NEWC = load_corpus([DB / "data/hang/chunks.jsonl"])                                   # 항 조각
O2N = json.load(open(DB / "data/hang/old_to_new.json", encoding="utf-8"))
ART = {k: v["article_id"] for k, v in O2N.items()} | {c["id"]: c["article_id"] for c in NEWC}   # 조각 → 조문
ARTICLES = {a["article_id"]: a for a in jl(DB / "data/hang/articles.jsonl")}                    # 조문 원문
NEW_ID = {c["id"]: c for c in NEWC}

NEW_BY_ART = defaultdict(list)          # 조문 → 항 조각 번호 (조문 안 순서)
for j, c in enumerate(NEWC):
    NEW_BY_ART[c["article_id"]].append(j)
OLD_ARTICLE = defaultdict(list)         # (문서, 조문) → 조문 조각 번호 (part 순). 별표 찾기에 씀
for i, c in enumerate(OLDC):
    if c["article"]:
        OLD_ARTICLE[(c["doc_id"], c["article"])].append(i)
for k in OLD_ARTICLE:
    OLD_ARTICLE[k].sort(key=lambda i: OLDC[i]["part"])

print(f"조문 조각 {len(OLDC):,} (평균 {int(np.mean([len(c['text']) for c in OLDC]))}자) · "
      f"항 조각 {len(NEWC):,} (평균 {int(np.mean([len(c['text']) for c in NEWC]))}자) · 조문 {len(NEW_BY_ART):,}")

## 3. 모델 · 색인 (BM25 + BGE-M3 임베딩은 조문 조각에만)

In [ ]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "없음 (느려요. 런타임 유형을 GPU로 바꾸세요)")

def _load(cls, name, **kw):
    try:
        return cls(name, device=DEVICE, model_kwargs={"torch_dtype": torch.float16 if HAS_GPU else torch.float32}, **kw)
    except TypeError:
        m = cls(name, device=DEVICE, **kw)
        if HAS_GPU:
            m.model.half() if hasattr(m, "model") else m.half()
        return m

kiwi = Kiwi()
_KEEP = {"NNG", "NNP", "NR", "SL", "SN", "SH", "XR", "VV", "VA"}
def ko_tokens(text):
    text = re.sub(r"\[[^\]]*\]", " ", text)           # [전화] 같은 마스킹 표시는 뺌
    return [t.form + ("다" if t.tag in ("VV", "VA") else "") for t in kiwi.tokenize(text) if t.tag in _KEEP]

EMBEDDER = _load(SentenceTransformer, CFG["embed_model"])
EMBEDDER.max_seq_length = 1024
RERANKER = _load(CrossEncoder, CFG["rerank_model"], max_length=1024)

t0 = time.time()
BM25 = BM25Okapi([ko_tokens(c["embed_text"]) for c in OLDC])
DOC_EMB = EMBEDDER.encode([c["embed_text"] for c in OLDC], batch_size=32, normalize_embeddings=True,
                          convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
print(f"BM25 + 임베딩 {time.time() - t0:.0f}초 · {DOC_EMB.shape}")

## 4. B 검색 함수
- `candidates(질문)` → ② 조문 조각 후보 40개 (순위순)
- `search_B(질문)` → ③ 조문 순서 + 항마다 리랭커 점수
- `context_B(…)` → LLM에 넘길 5개. 형식은 backend 검색 출력 약속(`{id, citation, revised, text, score}`)과 같아요

In [ ]:
_QCACHE, _RCACHE = {}, {}
RR = {"pairs": 0, "sec": 0.0}      # 속도 재기: 리랭커가 본 (질문, 항) 쌍 수와 걸린 시간

def embed_query(text):
    if text not in _QCACHE:
        _QCACHE[text] = EMBEDDER.encode([re.sub(r"\[[^\]]*\]", " ", text)], normalize_embeddings=True,
                                        convert_to_numpy=True)[0].astype(np.float32)
    return _QCACHE[text]

def bm25_rank(query, n):
    s = BM25.get_scores(ko_tokens(query))
    return [int(i) for i in np.argsort(-s)[:n] if s[i] > 0]

def dense_rank(query, n):
    return [int(i) for i in np.argsort(-(DOC_EMB @ embed_query(query)))[:n]]

def candidates(question):
    # ② 가중 RRF. 법령 조각은 law_weight 배
    n = CFG["k_candidates"]
    bq = be_retriever.rewrite(question) if CFG["rewrite_bm25"] else question
    lists = [(bm25_rank(bq, max(CFG["k_bm25"], n)), CFG["bm25_weight"]),
             (dense_rank(question, max(CFG["k_dense"], n)), CFG["dense_weight"])]
    score = defaultdict(float)
    for lst, w in lists:
        for r, i in enumerate(lst):
            score[i] += w / (CFG["rrf_c"] + r + 1)
    for i in score:
        if OLDC[i]["doc_kind"] == "법령":
            score[i] *= CFG["law_weight"]
    return sorted(score, key=score.get, reverse=True)[:n]

def rerank_hang(question, idxs, batch_size=16):
    # ③ {항 조각 번호: 0~1 점수} (+내부규정 가점)
    q = re.sub(r"\[[^\]]*\]", " ", question)
    todo = [j for j in idxs if (q, j) not in _RCACHE]
    if todo:
        t = time.perf_counter()
        s = np.asarray(RERANKER.predict([(q, NEWC[j]["embed_text"]) for j in todo], batch_size=batch_size,
                                        convert_to_numpy=True, show_progress_bar=False), dtype=float).reshape(-1)
        if s.min() < 0 or s.max() > 1:
            s = 1 / (1 + np.exp(-s))
        _RCACHE.update({(q, j): float(v) for j, v in zip(todo, s)})
        RR["pairs"] += len(todo)
        RR["sec"] += time.perf_counter() - t
    b = CFG["internal_bonus"]
    return {j: min(1.0, _RCACHE[(q, j)] + (b if NEWC[j]["doc_kind"] == "내부규정" else 0.0)) for j in idxs}

def search_B(question):
    # → (조문 순서, {항 번호: 점수})
    arts = list(dict.fromkeys(ART[OLDC[i]["id"]] for i in candidates(question) if OLDC[i]["id"] in ART))
    arts = [a for a in arts if NEW_BY_ART.get(a)]
    sc = rerank_hang(question, [j for a in arts for j in NEW_BY_ART[a]])
    art_score = {a: max(sc[j] for j in NEW_BY_ART[a]) for a in arts}
    return sorted(arts, key=lambda a: -art_score[a]), sc

def _head(c):
    return f"{c['article']}({c['article_title']})" if c["article_title"] else c["article"]

def context_B(art_rank, sc, k=None):
    # 조문마다 점수 높은 항부터 이어 붙임 (두 번째 항부터는 '제N조(제목)' 머리말을 뗌) + 별표 1개 → 1,200자
    out, cap = [], CFG["max_chars_per_article"]
    for a in art_rank[:k or CFG["k_context"]]:
        js = sorted(NEW_BY_ART[a], key=lambda j: -sc[j])
        parts = []
        for n, j in enumerate(js):
            t, h = NEWC[j]["text"], _head(NEWC[j])
            parts.append(t[len(h):].lstrip() if n and h and t.startswith(h) else t)
        text, c0 = "\n".join(parts), NEWC[js[0]]
        if CFG["attach_appendix"] and not c0["article"].startswith("별표"):
            for no in dict.fromkeys(re.findall(r"별표\s*(\d+)", ARTICLES.get(a, {}).get("text", text))):
                appx = OLD_ARTICLE.get((c0["doc_id"], f"별표 {no}"))
                if appx and ART.get(OLDC[appx[0]]["id"]) != a:
                    text += f"\n[참조 별표 {no}]\n" + "\n".join(OLDC[x]["text"] for x in appx)
                    break
        out.append({"id": a, "citation": c0["citation"], "revised": c0["revised"], "doc_kind": c0["doc_kind"],
                    "hang": [NEWC[j]["id"] for j in js], "score": round(sc[js[0]], 4),
                    "text": text if len(text) <= cap else text[:cap] + " …(생략)"})
    return out

def search(question, k=None):
    # backend 검색과 같은 모양: [{id, citation, revised, text, score}] 높은 순
    return context_B(*search_B(question), k)

## 5. 질문 넣어 보기
`ask("질문")` → LLM에 넘어갈 조문 5개, 점수, 항 순서, 넘기는 글. `full=True`면 글 전체를 보여 줘요.

In [ ]:
def ask(question, k=5, full=False):
    t = time.perf_counter()
    hits = search(question, k)
    ms = (time.perf_counter() - t) * 1000
    print(f"Q. {question}   ({ms:.0f}ms · 검색어 보강: {be_retriever.rewrite(question)})")
    for n, h in enumerate(hits, 1):
        print(f"\n[{n}] {h['score']:.3f}  {h['citation']}  ({h['doc_kind']} · 항 {len(h['hang'])}개 · {len(h['text'])}자)")
        print("    " + (h["text"] if full else h["text"][:200].replace("\n", " ") + ("…" if len(h["text"]) > 200 else "")))
    return hits

ask("결혼하면 휴가 며칠이야?");

In [ ]:
ask("며느리가 출산하면 할아버지인 저도 휴가 있어요?");
ask("광역시 1박 2일 출장비 얼마야?");

## 6. 골든셋 지표 (dev 75 · test 166)
- **정답 항 전달률**: 골든셋 정답 항(`gold_hang`)의 **앞 30자·끝 30자가 모두** LLM에 넘기는 5개 안에 있는 비율. 중간에 잘려도 실패
- **조문 recall@5·30 / MRR**: 조문 순서에서 정답 조문(`gold_articles`) 위치
- **긴 조문 문항**: 정답 조문 원문이 1,200자를 넘는 문항만 따로
- **속도**: 캐시를 비우고 질문마다 처음부터 (질문 임베딩 → ② → ③ → 넘길 글). GPU 첫 호출은 빼고 잼
- 참고 (11/19 L4): 정답 항 전달률 dev 0.743 · test 0.735, 조문 recall@5 0.65~0.69, p50 632ms · p95 1,662ms

In [ ]:
def squash(s): return re.sub(r"\s+", "", s)

def hang_sig(cid):
    c = NEW_ID[cid]
    body, h = c["text"], _head(c)
    s = squash(body[len(h):] if h and body.startswith(h) else body)
    return (s[:30], s[-30:]) if len(s) > 30 else (s, s)

def delivered(cid, blob):
    a, b = hang_sig(cid)
    return a in blob and b in blob

def art_metrics(gold, arts):
    g = set(gold)
    hits = [i for i, x in enumerate(arts) if x in g]
    return {"조문 recall@5": sum(i < 5 for i in hits) / len(g), "조문 recall@30": sum(i < 30 for i in hits) / len(g),
            "조문 MRR": 1 / (hits[0] + 1) if hits else 0.0}

def sync():
    if HAS_GPU:
        torch.cuda.synchronize()

search("연차는 며칠이야?")       # 예열
ROWS = []
for split in ("dev", "test"):
    QS = [it for it in ITEMS if SPLIT[it["id"]] == split and it["gold_chunks"]["primary"]]
    t0 = time.time()
    for it in QS:
        _QCACHE.clear(); _RCACHE.clear(); RR.update(pairs=0, sec=0.0)
        sync(); tq = time.perf_counter()
        arts, sc = search_B(it["question"])
        ctx = context_B(arts, sc)
        sync(); ms = (time.perf_counter() - tq) * 1000
        blob = squash("\n".join(h["text"] for h in ctx))
        gh = it["gold_hang"]
        hang_ok = gh.get("auto") is True and bool(gh.get("primary"))
        ROWS.append({"split": split, "id": it["id"], "question": it["question"],
                     "long": any(len(ARTICLES.get(a, {}).get("text", "")) > CFG["max_chars_per_article"]
                                 for a in it["gold_articles"]["primary"]),
                     "정답 항 전달률": np.mean([delivered(c, blob) for c in gh["primary"]]) if hang_ok else np.nan,
                     **art_metrics(it["gold_articles"]["primary"], arts),
                     "LLM 입력 글자수": sum(len(h["text"]) for h in ctx), "잘린 조문 수": sum(h["text"].endswith("…(생략)") for h in ctx),
                     "ms": ms, "리랭커 ms": RR["sec"] * 1000, "리랭커 쌍 수": RR["pairs"], "top": ctx[0]["citation"] if ctx else ""})
    print(f"{split}: {len(QS)}문항 {time.time() - t0:.0f}초")
R = pd.DataFrame(ROWS)

KEYS = ["정답 항 전달률", "조문 recall@5", "조문 recall@30", "조문 MRR", "LLM 입력 글자수", "잘린 조문 수"]
def summary(df):
    s = df[KEYS].mean().round(3)
    s["문항"], s["항 정답 있는 문항"] = len(df), df["정답 항 전달률"].count()
    return s

TAB = pd.DataFrame({f"{sp} {name}": summary(D) for sp in ("dev", "test")
                    for name, D in (("전체", R[R["split"] == sp]), ("긴 조문", R[(R["split"] == sp) & R["long"]]))})
display(TAB)

g = R.groupby("split")
SPEED = pd.DataFrame({"p50 ms": g["ms"].median(), "p95 ms": g["ms"].quantile(0.95), "평균 ms": g["ms"].mean(),
                      "리랭커 평균 ms": g["리랭커 ms"].mean(), "리랭커 쌍 수 평균": g["리랭커 쌍 수"].mean(),
                      "리랭커 쌍 수 최대": g["리랭커 쌍 수"].max()}).round(1)
print("속도 · GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "cpu")
display(SPEED)

In [ ]:
# 정답 항이 LLM까지 못 간 문항 (test) — 조문을 못 찾았나, 찾았는데 1,200자에서 잘렸나
miss = R[(R["split"] == "test") & (R["정답 항 전달률"] < 1)].copy()
miss["원인"] = np.where(miss["조문 recall@5"] == 0, "조문 못 찾음", np.where(miss["잘린 조문 수"] > 0, "잘림 가능", "다른 항만 감"))
print(miss["원인"].value_counts().to_dict())
pd.set_option("display.max_colwidth", 60)
display(miss[["id", "question", "정답 항 전달률", "조문 recall@5", "원인", "top"]].head(30))

## 7. 거절 기준값 (`verify_min_score`)
리랭커 1등 점수가 이 값보다 낮으면 "규정에서 확인되지 않아요"로 거절해요.
- dev **답할 수 있는 문항**의 1등 점수 아래 5% 지점으로 정해요 (잘못된 거절 ≤ 5%) → test로 확인
- ⚠️ '규정 없음' 문항이 dev 3 · test 7개뿐이라 거절률은 참고만. **AUC**(1등 점수로 답할 문항과 규정 없음 문항을 가르는 힘, 1이면 완벽 · 0.5면 못 가름)를 같이 봐요

In [ ]:
def auc(pos, neg):
    if not pos or not neg:
        return np.nan
    return float(np.mean([(p > n) + 0.5 * (p == n) for p in pos for n in neg]))

TOP = []
for it in ITEMS:
    if SPLIT[it["id"]] not in ("dev", "test"):
        continue
    arts, sc = search_B(it["question"])
    TOP.append({"split": SPLIT[it["id"]], "id": it["id"], "type": it["type"],
                "top": max(sc[j] for j in NEW_BY_ART[arts[0]]) if arts else 0.0})
TOP = pd.DataFrame(TOP)
ans = lambda sp: TOP[(TOP["split"] == sp) & (TOP["type"] != "규정 없음")]["top"]
neg = lambda sp: TOP[(TOP["split"] == sp) & (TOP["type"] == "규정 없음")]["top"]
THR = round(float(np.quantile(ans("dev"), 0.05)), 3)
print(f"verify_min_score (dev로 정함) = {THR}")
display(pd.DataFrame([{"split": sp, "잘못된 거절률 (목표 ≤0.05)": round((ans(sp) < THR).mean(), 3),
                       "규정 없음 거절률 (목표 ≥0.9)": round((neg(sp) < THR).mean(), 3), "규정 없음 문항": len(neg(sp)),
                       "규정 없음 1등 점수": ", ".join(f"{s:.2f}" for s in sorted(neg(sp)))} for sp in ("dev", "test")]))
print("AUC (dev+test):", round(auc(list(TOP[TOP["type"] != "규정 없음"]["top"]), list(TOP[TOP["type"] == "규정 없음"]["top"])), 3))
display(TOP.groupby("type")["top"].describe().round(3))

## 8. 결과 저장 (JSON 다운로드)

In [ ]:
from google.colab import files
out = {"cfg": CFG, "gpu": torch.cuda.get_device_name(0) if HAS_GPU else "cpu",
       "summary": TAB.to_dict(), "speed": SPEED.to_dict(), "verify_min_score": THR,
       "repos": {r: subprocess.run(f"git -C {ROOT / r} log -1 --format=%h", shell=True, capture_output=True, text=True).stdout.strip()
                 for r in ("backend", "DB", "RAG")},
       "rows": json.loads(R.to_json(orient="records", force_ascii=False)),
       "top_scores": TOP.to_dict(orient="records")}
fn = "b_search_eval.json"
json.dump(out, open(fn, "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
files.download(fn)